In [20]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [79]:
# load dataset and take a random sample
df = pd.read_csv("data/Reviews.csv")
df_sample = df.sample(n=50000, random_state=42)
print(f"Sample size: {len(df_sample)}")

Sample size: 50000


In [81]:
# keep only ratings of 4 or 5 (positive interactions)
df_sample = df_sample[df_sample["Score"] >= 4]
print(f"After keeping positive reviews: {len(df_sample)}")

After keeping positive reviews: 39105


In [83]:
# remove users who reviewed fewer than 2 products
user_counts = df_sample["UserId"].value_counts()
valid_users = user_counts[user_counts >= 2].index
df_sample = df_sample[df_sample["UserId"].isin(valid_users)]
print(f"After removing users with < 2 products: {len(df_sample)}")

After removing users with < 2 products: 11330


In [85]:
# remove products that appear fewer than 4 times
product_counts = df_sample["ProductId"].value_counts()
valid_products = product_counts[product_counts >= 4].index
df_sample = df_sample[df_sample["ProductId"].isin(valid_products)]
print(f"After removing rare products: {len(df_sample)}")
print(f"Unique users: {df_sample['UserId'].nunique()}")
print(f"Unique products: {df_sample['ProductId'].nunique()}")

After removing rare products: 5460
Unique users: 2755
Unique products: 704


In [87]:
# group products by user - each user = one transaction
transactions = df_sample.groupby("UserId")["ProductId"].apply(list).tolist()

# keep only transactions with at least 2 products
transactions = [t for t in transactions if len(t) >= 2]

lengths = [len(t) for t in transactions]
print(f"Total transactions: {len(transactions)}")
print(f"Average products per transaction: {sum(lengths)/len(lengths):.2f}")
print(f"Max products in a transaction: {max(lengths)}")
print(f"Min products in a transaction: {min(lengths)}")

Total transactions: 1883
Average products per transaction: 2.44
Max products in a transaction: 14
Min products in a transaction: 2


In [89]:
# convert transactions into a true/false table
# each row = one user, each column = one product
# true means the user bought that product, false means they didn't
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_trans = pd.DataFrame(te_array, columns=te.columns_)

print(f"Transaction matrix shape: {df_trans.shape}")
df_trans.head()

Transaction matrix shape: (1883, 688)


,7310172001,7310172101,B000084ETV,B000084EZ4,B00008CQVA,B00014DXCC,B0001ES9F8,B00020HHAO,B00020HHE0,B00020HHGS,...,B008O3G2GG,B008RWUHA6,B008RWUKXK,B008ZRKZSM,B0090X8IPM,B00954NY46,B00954NYVY,B0096EZHM2,B009E7YC54,B009GHI5Q4
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [107]:
# find frequent itemsets using apriori
# min_support=0.002 means a product combo must appear in at least 0.2% of transactions
frequent_itemsets = apriori(df_trans, min_support=0.002, use_colnames=True)

print(f"Total frequent itemsets found: {len(frequent_itemsets)}")
frequent_itemsets.head()

Total frequent itemsets found: 657


,support,itemsets
0,0.006373,frozenset({7310172001})
1,0.002655,frozenset({7310172101})
2,0.003186,frozenset({B000084ETV})
3,0.005311,frozenset({B000084EZ4})
4,0.002655,frozenset({B00008CQVA})


In [109]:
# generate association rules from frequent itemsets
# min_threshold=0.1 means confidence must be at least 10%
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)

# keep only rules with lift > 1 (meaningful relationships)
rules = rules[rules["lift"] >= 1.0]

print(f"Total rules generated: {len(rules)}")
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head()


Total rules generated: 260


,antecedents,consequents,support,confidence,lift
0,frozenset({7310172001}),frozenset({B001B4VOQI}),0.002124,0.333333,62.766667
1,frozenset({B001B4VOQI}),frozenset({7310172001}),0.002124,0.400000,62.766667
2,frozenset({B000WFORH0}),frozenset({B000084EZ4}),0.002124,0.571429,107.600000
3,frozenset({B000084EZ4}),frozenset({B000WFORH0}),0.002124,0.400000,107.600000
4,frozenset({B00014DXCC}),frozenset({B00412W76S}),0.002124,0.222222,32.188034


In [111]:
# convert frozensets to lists for easier handling later
rules_simple = rules.copy()
rules_simple["antecedents"] = rules_simple["antecedents"].apply(lambda x: list(x))
rules_simple["consequents"] = rules_simple["consequents"].apply(lambda x: list(x))

print(f"Total rules: {len(rules_simple)}")
rules_simple[["antecedents", "consequents", "support", "confidence", "lift"]].head()

Total rules: 260


,antecedents,consequents,support,confidence,lift
0,[7310172001],[B001B4VOQI],0.002124,0.333333,62.766667
1,[B001B4VOQI],[7310172001],0.002124,0.400000,62.766667
2,[B000WFORH0],[B000084EZ4],0.002124,0.571429,107.600000
3,[B000084EZ4],[B000WFORH0],0.002124,0.400000,107.600000
4,[B00014DXCC],[B00412W76S],0.002124,0.222222,32.188034


In [113]:
# combine summary and text into one content field per product
# we aggregate all reviews for each product into one document
product_df = df_sample.groupby("ProductId").agg({"Summary": " ".join, "Text": " ".join}).reset_index()
product_df["content"] = product_df["Summary"] + " " + product_df["Text"]

print(f"Total products: {len(product_df)}")
product_df.head()

Total products: 704


,ProductId,Summary,Text,content
0,7310172001,Like Candy for Your Dog dogs love them Healthy...,Dogs love these treats more than any other tre...,Like Candy for Your Dog dogs love them Healthy...
1,7310172101,Corgi Cocaine Great for puppy training THE BES...,We cut these up into small pieces for our Corg...,Corgi Cocaine Great for puppy training THE BES...
2,B00004RYGX,"Good Movie For Halloween Wild, Crazy Fun With ...",This is a good classic movie to put on at Hall...,"Good Movie For Halloween Wild, Crazy Fun With ..."
3,B000084ETV,Great for big dogs Fantastic food! Feeding it ...,I have a four month old st Bernard puppy and h...,Great for big dogs Fantastic food! Feeding it ...
4,B000084EZ4,The best cat food on the planet Great! The Bes...,I have a seven year old gray tabby named Buddy...,The best cat food on the planet Great! The Bes...


In [115]:
# create tf-idf vectors from product content
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(product_df["content"])

# compute cosine similarity between all products
cosine_sim = cosine_similarity(tfidf_matrix)

# convert to dataframe for easy lookup
cosine_df = pd.DataFrame(cosine_sim, index=product_df["ProductId"], columns=product_df["ProductId"])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Cosine similarity matrix shape: {cosine_df.shape}")

TF-IDF matrix shape: (704, 12364)
Cosine similarity matrix shape: (704, 704)


In [117]:
# view the cosine similarity matrix
print("Cosine similarity matrix:")
print(cosine_df.shape)
cosine_df

Cosine similarity matrix:
(704, 704)


ProductId,7310172001,7310172101,B00004RYGX,B000084ETV,B000084EZ4,B00008CQVA,B00014DXCC,B0001ES9F8,B00020HHAO,B00020HHE0,...,B008O3G2GG,B008RWUHA6,B008RWUKXK,B008ZRKZSM,B0090X8IPM,B00954NY46,B00954NYVY,B0096EZHM2,B009E7YC54,B009GHI5Q4
ProductId,,,,,,,,,,,,,,,,,,,,,
7310172001,1.000000,0.609813,0.032619,0.214033,0.101394,0.092294,0.037795,0.037603,0.040131,0.040485,...,0.258069,0.097339,0.067144,0.032090,0.047858,0.028117,0.020365,0.062155,0.031047,0.051448
7310172101,0.609813,1.000000,0.013879,0.131534,0.058019,0.058300,0.023946,0.022607,0.029726,0.021579,...,0.142023,0.048569,0.040468,0.038876,0.032720,0.018645,0.022164,0.044179,0.036944,0.079717
B00004RYGX,0.032619,0.013879,1.000000,0.030811,0.070370,0.065149,0.043248,0.029685,0.032033,0.040102,...,0.051657,0.085633,0.056162,0.012225,0.046763,0.027111,0.014059,0.045251,0.028690,0.036579
B000084ETV,0.214033,0.131534,0.030811,1.000000,0.214269,0.182011,0.059601,0.031465,0.049638,0.027481,...,0.115827,0.083973,0.066957,0.030841,0.036241,0.031924,0.027036,0.107150,0.034240,0.186092
B000084EZ4,0.101394,0.058019,0.070370,0.214269,1.000000,0.737027,0.075748,0.056431,0.080120,0.070155,...,0.102635,0.211835,0.153822,0.048508,0.085904,0.042432,0.033759,0.387511,0.062639,0.401059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
B00954NY46,0.028117,0.018645,0.027111,0.031924,0.042432,0.040550,0.054915,0.225534,0.050907,0.031198,...,0.020402,0.059051,0.050733,0.024439,0.324617,1.000000,0.511534,0.031873,0.027628,0.023779
B00954NYVY,0.020365,0.022164,0.014059,0.027036,0.033759,0.041989,0.057914,0.242509,0.051373,0.046649,...,0.027485,0.053498,0.047396,0.020994,0.492148,0.511534,1.000000,0.034984,0.031108,0.027044
B0096EZHM2,0.062155,0.044179,0.045251,0.107150,0.387511,0.368869,0.039369,0.045786,0.044950,0.040015,...,0.074826,0.109904,0.079129,0.017535,0.052962,0.031873,0.034984,1.000000,0.045031,0.260686


In [119]:
# show the top 5 most similar products to the first product
first_product = product_df["ProductId"].iloc[0]
print(f"Products most similar to {first_product}:")
cosine_df[first_product].sort_values(ascending=False).head(6)

Products most similar to 7310172001:


ProductId
7310172001    1.000000
B0002DGRQ6    0.767451
B001B4VOQI    0.753460
B000255OIG    0.719575
B0002DGRSY    0.696157
B0007A0AQW    0.652591
Name: 7310172001, dtype: float64

In [121]:
# build a dictionary for quick association rule lookup
# ar_dict[product] = {candidate: confidence}
ar_dict = {}

for _, row in rules_simple.iterrows():
    for antecedent in row["antecedents"]:
        for consequent in row["consequents"]:
            if antecedent not in ar_dict:
                ar_dict[antecedent] = {}
            # keep the highest confidence if multiple rules exist
            if consequent not in ar_dict[antecedent]:
                ar_dict[antecedent][consequent] = row["confidence"]
            else:
                ar_dict[antecedent][consequent] = max(ar_dict[antecedent][consequent], row["confidence"])

print(f"Number of products with AR rules: {len(ar_dict)}")

Number of products with AR rules: 146


In [125]:
# function to get similar products based on content (tf-idf)
def get_similar_products(product_id, top_n=10):
    # check if product exists in our cosine similarity matrix
    if product_id not in cosine_df.index:
        return []
    
    # get similarity scores for this product with all others
    # sort by highest similarity and skip the product itself (index 0)
    similar = cosine_df[product_id].sort_values(ascending=False).iloc[1:top_n+1]
    
    # return list of (product_id, cosine_score) tuples
    return list(zip(similar.index, similar.values))

In [127]:
# test the function with the first product
test_product = product_df["ProductId"].iloc[0]
print(f"Top 5 similar products to {test_product}:")
results = get_similar_products(test_product, top_n=5)
for product, score in results:
    print(f"  {product}  cosine={score:.3f}")

Top 5 similar products to 7310172001:
  B0002DGRQ6  cosine=0.767
  B001B4VOQI  cosine=0.753
  B000255OIG  cosine=0.720
  B0002DGRSY  cosine=0.696
  B0007A0AQW  cosine=0.653


In [129]:
# function to get hybrid recommendations for a product
def hybrid_recommend(product_id, alpha=0.5, top_n=5):
    scores = {}
    
    # --- association rules component ---
    if product_id in ar_dict:
        for candidate, confidence in ar_dict[product_id].items():
            scores[candidate] = scores.get(candidate, 0) + alpha * confidence
    
    # --- tf-idf content component ---
    similar_products = get_similar_products(product_id, top_n=20)
    for candidate, cosine_score in similar_products:
        scores[candidate] = scores.get(candidate, 0) + (1 - alpha) * cosine_score
    
    # --- remove the input product itself from recommendations ---
    scores.pop(product_id, None)
    
    # --- sort by highest hybrid score and return top n ---
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

In [131]:
# test hybrid recommendation with alpha=0.5
test_product = product_df["ProductId"].iloc[0]
print(f"Hybrid recommendations for product: {test_product} (alpha=0.5)")
print("-" * 50)
results = hybrid_recommend(test_product, alpha=0.5, top_n=5)
for i, (product, score) in enumerate(results, 1):
    print(f"{i}. {product}  hybrid_score={score:.3f}")

Hybrid recommendations for product: 7310172001 (alpha=0.5)
--------------------------------------------------
1. B001B4VOQI  hybrid_score=0.543
2. B0002DGRQ6  hybrid_score=0.384
3. B000255OIG  hybrid_score=0.360
4. B0002DGRSY  hybrid_score=0.348
5. B0007A0AQW  hybrid_score=0.326


In [133]:
def evaluate_system(transactions, alpha=0.5, k=5):
    hits = 0
    total = 0
    
    for t in transactions:
        # skip transactions with less than 2 products
        if len(t) < 2:
            continue
        
        # leave-one-out: hide the last product as ground truth
        input_item = t[0]
        test_item = t[-1]
        
        # get hybrid recommendations
        recs = hybrid_recommend(input_item, alpha=alpha, top_n=k)
        rec_products = [r[0] for r in recs]
        
        # check if ground truth is in recommendations
        if test_item in rec_products:
            hits += 1
        
        total += 1
    
    precision = hits / (total * k) if total > 0 else 0
    recall = hits / total if total > 0 else 0
    
    return precision, recall

In [135]:
# test different alpha values
print("Alpha --> Precision, Recall")
print("-" * 40)
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    p, r = evaluate_system(transactions, alpha=alpha, k=5)
    print(f"Alpha={alpha} --> Precision={p:.3f}, Recall={r:.3f}")

Alpha --> Precision, Recall
----------------------------------------
Alpha=0.0 --> Precision=0.119, Recall=0.597
Alpha=0.25 --> Precision=0.122, Recall=0.609
Alpha=0.5 --> Precision=0.123, Recall=0.613
Alpha=0.75 --> Precision=0.123, Recall=0.613
Alpha=1.0 --> Precision=0.123, Recall=0.613


In [145]:
# generate 6 sample recommendation outputs
sample_products = [product_df["ProductId"].iloc[0],
                   product_df["ProductId"].iloc[10],
                   product_df["ProductId"].iloc[20],
                   product_df["ProductId"].iloc[30],
                   product_df["ProductId"].iloc[50],
                   product_df["ProductId"].iloc[70]]

for i, product_id in enumerate(sample_products, 1):
    print("=" * 50)
    print(f"Sample {i}")
    print(f"Seed product: {product_id}")
    
    # ar candidates
    print("\n--- Association Rules Candidates ---")
    if product_id in ar_dict:
        ar_candidates = sorted(ar_dict[product_id].items(), 
                               key=lambda x: x[1], reverse=True)[:3]
        for candidate, confidence in ar_candidates:
            print(f"  {candidate}  conf={confidence:.3f}")
    else:
        print("  No AR rules found for this product")
    
    # tfidf candidates
    print("\n--- TF-IDF Content Candidates ---")
    tfidf_candidates = get_similar_products(product_id, top_n=3)
    for candidate, cosine in tfidf_candidates:
        print(f"  {candidate}  cosine={cosine:.3f}")
    
    # hybrid recommendations
    print(f"\n--- Hybrid Top-5 Recommendations (alpha=0.5) ---")
    recs = hybrid_recommend(product_id, alpha=0.5, top_n=5)
    for j, (candidate, score) in enumerate(recs, 1):
        print(f"  {j}. {candidate}  hybrid_score={score:.3f}")
    print()

Sample 1
Seed product: 7310172001

--- Association Rules Candidates ---
  B001B4VOQI  conf=0.333

--- TF-IDF Content Candidates ---
  B0002DGRQ6  cosine=0.767
  B001B4VOQI  cosine=0.753
  B000255OIG  cosine=0.720

--- Hybrid Top-5 Recommendations (alpha=0.5) ---
  1. B001B4VOQI  hybrid_score=0.543
  2. B0002DGRQ6  hybrid_score=0.384
  3. B000255OIG  hybrid_score=0.360
  4. B0002DGRSY  hybrid_score=0.348
  5. B0007A0AQW  hybrid_score=0.326

Sample 2
Seed product: B00020HHGS

--- Association Rules Candidates ---
  No AR rules found for this product

--- TF-IDF Content Candidates ---
  B0006I5M2M  cosine=0.687
  B00412W76S  cosine=0.600
  B001GCTTRQ  cosine=0.554

--- Hybrid Top-5 Recommendations (alpha=0.5) ---
  1. B0006I5M2M  hybrid_score=0.344
  2. B00412W76S  hybrid_score=0.300
  3. B001GCTTRQ  hybrid_score=0.277
  4. B000FVBYCW  hybrid_score=0.277
  5. B007RLRCLK  hybrid_score=0.258

Sample 3
Seed product: B0002DGRZC

--- Association Rules Candidates ---
  No AR rules found for this